# Murmur 350M — mixed English/Russian/code/math training

This notebook builds the mixed corpus automatically from public Hugging Face streams. No external dataset attachment is required. The model remains the GQA baseline; SISO and MIMO have separate smoke notebooks.

In [ ]:
from pathlib import Path
import subprocess, sys, torch
REPO = Path('/workspace/murmur-science')
if not REPO.exists():
    subprocess.run(['git','clone','--branch','codex/dataset-mix-notebooks','https://github.com/orkrs/murmur-science.git',str(REPO)], check=True)
%cd /workspace/murmur-science
sys.path.insert(0, str(Path.cwd() / 'src'))
print('repo ready:', Path.cwd())

In [ ]:
%pip install -q datasets sentencepiece pyarrow pandas einops ninja
if not torch.cuda.is_available(): raise RuntimeError('CUDA GPU is required')
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
!python scripts/param_count.py --config configs/mixed_350m.toml
from murmur.config import load_run_config
config = load_run_config(Path('configs/mixed_350m.toml'))
assert config.model.mixer == 'gqa'
assert config.train.max_tokens == 7_500_000_000
print('Verified: 353.5M-class GQA baseline; 7.5B-token mixed run.')

In [ ]:
# Automatic token-balanced dataset build: FineWeb-Edu + Russian + code + math
subprocess.run([sys.executable, 'scripts/build_hf_mix.py', '--profile', 'mixed_350m', '--output', 'artifacts/mixed_350m_corpus', '--max-tokens', '7500000000'], check=True)
report = __import__('json').loads(Path('artifacts/mixed_350m_corpus/provenance.json').read_text(encoding='utf-8'))
assert report['estimated_tokens'] > 0 and Path('artifacts/mixed_350m_corpus/train.jsonl').exists()
print(report)

In [ ]:
!python scripts/train_tokenizer.py --corpus artifacts/mixed_350m_corpus/corpus.txt --output artifacts/mixed_350m_tokenizer.model --vocab-size 48000
!python scripts/prepare_data.py --config configs/mixed_350m.toml --tokenizer artifacts/mixed_350m_tokenizer.model --train-input artifacts/mixed_350m_corpus/train.jsonl --val-input artifacts/mixed_350m_corpus/val.jsonl --output artifacts/mixed_350m_data
assert list(Path('artifacts/mixed_350m_data').glob('train_*.bin')) and list(Path('artifacts/mixed_350m_data').glob('val_*.bin'))
print('Mixed shards ready')

In [ ]:
run_dir = Path('artifacts/runs/murmur_350m_mixed')
subprocess.run([sys.executable, 'scripts/train.py', '--config', 'configs/mixed_350m.toml', '--run-dir', str(run_dir), '--device', 'cuda'], check=True)
assert (run_dir / 'checkpoints' / 'last' / 'COMPLETED').exists()
print('350M mixed training checkpoint ready')

In [ ]:
subprocess.run([sys.executable, 'scripts/evaluate.py', '--config', 'configs/mixed_350m.toml', '--checkpoint', 'artifacts/runs/murmur_350m_mixed/checkpoints/last', '--output', 'artifacts/mixed_350m_eval.json', '--device', 'cuda'], check=True)
print(Path('artifacts/mixed_350m_eval.json').read_text(encoding='utf-8'))